# 01 — Binomial No-Arbitrage Replication

## Purpose

This notebook rebuilds the derivative-pricing project from first principles.

The goal is not to forecast the underlying asset. The goal is to show how a derivative price can be determined by **no-arbitrage replication**.

This is the starting point before Black-Scholes, stochastic calculus, implied volatility, volatility surfaces, Heston, Bates, or calibration.

---

## Core question

> In a simple binomial market, how can an option price be determined without using the real-world probability of an up move?

---

## Main idea

A derivative can be priced by constructing a portfolio of the stock and the risk-free asset that exactly replicates the derivative payoff in every future state.

If two assets have the same payoff in all future states, they must have the same price today. Otherwise, an arbitrage opportunity exists.

---

## What this notebook covers

1. One-period binomial stock model  
2. No-arbitrage bounds  
3. Replicating portfolio  
4. Risk-neutral probability  
5. European call and put pricing  
6. Put-call parity  
7. Two-period binomial extension  
8. Convergence intuition toward continuous-time pricing  

---

## What this notebook does not prove

This notebook does **not** prove Black-Scholes.

It does **not** use Brownian motion, Ito calculus, stochastic differential equations, or volatility surfaces.

It does **not** claim anything about real SPY option prices.

It only establishes the first principle:

> Derivative pricing begins with replication and no-arbitrage, not prediction.

---

## Pass condition

This notebook passes only if the option price is derived from replication/no-arbitrage logic rather than from expected payoff under the real-world probability.

In [1]:
# ============================================================
# 01 — Binomial No-Arbitrage Replication
# Setup cell
# ============================================================

from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import math

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Notebook display settings
# ------------------------------------------------------------

pd.set_option("display.float_format", "{:,.6f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


# ------------------------------------------------------------
# Numerical tolerance
# ------------------------------------------------------------

TOL = 1e-10


# ------------------------------------------------------------
# Type aliases
# ------------------------------------------------------------

OptionType = Literal["call", "put"]


# ------------------------------------------------------------
# Small validation helpers
# ------------------------------------------------------------

def assert_positive(name: str, value: float) -> None:
    """Raise an error if a scalar parameter is not strictly positive."""
    if not math.isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be positive and finite. Got {value!r}.")


def assert_nonnegative(name: str, value: float) -> None:
    """Raise an error if a scalar parameter is negative."""
    if not math.isfinite(value) or value < 0:
        raise ValueError(f"{name} must be nonnegative and finite. Got {value!r}.")


def assert_between(name: str, value: float, lower: float, upper: float) -> None:
    """Raise an error if value is outside [lower, upper]."""
    if not math.isfinite(value) or not (lower <= value <= upper):
        raise ValueError(f"{name} must be in [{lower}, {upper}]. Got {value!r}.")


print("Setup complete.")

Setup complete.


## 1. One-period binomial market

We begin with the smallest possible market.

There is one stock and one risk-free asset.

The stock price today is $S_0$. After one period, the stock can move to only two possible values:

$$
S_u = uS_0
$$

$$
S_d = dS_0
$$

where:

- $u > 1$ is the up multiplier
- $0 < d < 1$ is the down multiplier
- $S_u$ is the stock price in the up state
- $S_d$ is the stock price in the down state

The risk-free asset grows by the gross return:

$$
R = 1 + r
$$

where $r$ is the one-period risk-free rate.

For a no-arbitrage binomial model, the risk-free return must lie between the down and up stock returns:

$$
d < R < u
$$

If this condition fails, one asset dominates the other and arbitrage becomes possible.

The derivative payoff is computed in each future state. For a European call option:

$$
C_u = \max(S_u - K, 0)
$$

$$
C_d = \max(S_d - K, 0)
$$

For a European put option:

$$
P_u = \max(K - S_u, 0)
$$

$$
P_d = \max(K - S_d, 0)
$$

The task is to find the option value today without forecasting whether the up or down state is more likely.

In [2]:
# ============================================================
# One-period binomial market primitives
# ============================================================

@dataclass(frozen=True)
class OnePeriodBinomialMarket:
    """
    One-period binomial market.

    Parameters
    ----------
    s0:
        Initial stock price.
    u:
        Up-state stock multiplier.
    d:
        Down-state stock multiplier.
    r:
        One-period risk-free rate.

    Notes
    -----
    The gross risk-free return is R = 1 + r.

    The no-arbitrage condition is:

        d < R < u

    This means the risk-free asset return must lie strictly between
    the down-state and up-state stock returns.
    """

    s0: float
    u: float
    d: float
    r: float

    def __post_init__(self) -> None:
        assert_positive("s0", self.s0)
        assert_positive("u", self.u)
        assert_positive("d", self.d)
        assert_nonnegative("r", self.r)

        if not self.d < 1 < self.u:
            raise ValueError(
                "Expected stock multipliers to satisfy d < 1 < u. "
                f"Got d={self.d}, u={self.u}."
            )

    @property
    def gross_risk_free_return(self) -> float:
        """Gross one-period risk-free return."""
        return 1.0 + self.r

    @property
    def stock_up(self) -> float:
        """Stock price in the up state."""
        return self.u * self.s0

    @property
    def stock_down(self) -> float:
        """Stock price in the down state."""
        return self.d * self.s0

    @property
    def is_no_arbitrage(self) -> bool:
        """Return True if the one-period no-arbitrage condition holds."""
        return self.d < self.gross_risk_free_return < self.u

    def validate_no_arbitrage(self) -> None:
        """Raise an error if the binomial market violates no-arbitrage."""
        R = self.gross_risk_free_return

        if not self.is_no_arbitrage:
            raise ValueError(
                "No-arbitrage condition failed. Expected d < R < u. "
                f"Got d={self.d:.6f}, R={R:.6f}, u={self.u:.6f}."
            )

    def state_table(self) -> pd.DataFrame:
        """Return a table of stock values across future states."""
        return pd.DataFrame(
            {
                "state": ["up", "down"],
                "stock_multiplier": [self.u, self.d],
                "stock_price": [self.stock_up, self.stock_down],
            }
        )


def european_option_payoff(
    stock_price: float,
    strike: float,
    option_type: OptionType,
) -> float:
    """
    European call or put payoff at maturity.
    """
    assert_positive("stock_price", stock_price)
    assert_positive("strike", strike)

    if option_type == "call":
        return max(stock_price - strike, 0.0)

    if option_type == "put":
        return max(strike - stock_price, 0.0)

    raise ValueError(f"Unsupported option_type: {option_type!r}")


def option_state_payoffs(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
) -> pd.DataFrame:
    """
    Return the option payoff in each future state.
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)

    states = market.state_table()
    states["strike"] = strike
    states["option_type"] = option_type
    states["option_payoff"] = states["stock_price"].map(
        lambda s: european_option_payoff(s, strike, option_type)
    )

    return states


# ------------------------------------------------------------
# Baseline example
# ------------------------------------------------------------

market = OnePeriodBinomialMarket(
    s0=100.0,
    u=1.20,
    d=0.80,
    r=0.05,
)

strike = 100.0
option_type: OptionType = "call"

market.validate_no_arbitrage()

display(market.state_table())
display(option_state_payoffs(market, strike, option_type))

,state,stock_multiplier,stock_price
0,up,1.200000,120.000000
1,down,0.800000,80.000000


,state,stock_multiplier,stock_price,strike,option_type,option_payoff
0,up,1.200000,120.000000,100.000000,call,20.000000
1,down,0.800000,80.000000,100.000000,call,0.000000


## 2. Replicating portfolio

Now we price the option by replication.

We construct a portfolio with:

- $\Delta$ shares of stock
- $B$ dollars invested in the risk-free asset today

The value of this portfolio today is:

$$
V_0 = \Delta S_0 + B
$$

After one period, the risk-free position grows from $B$ to $RB$, where:

$$
R = 1 + r
$$

The portfolio value in the up state is:

$$
V_u = \Delta S_u + RB
$$

The portfolio value in the down state is:

$$
V_d = \Delta S_d + RB
$$

To replicate the option, the portfolio must match the option payoff in both states:

$$
\Delta S_u + RB = C_u
$$

$$
\Delta S_d + RB = C_d
$$

Subtracting the down-state equation from the up-state equation gives:

$$
\Delta(S_u - S_d) = C_u - C_d
$$

So the stock holding is:

$$
\Delta = \frac{C_u - C_d}{S_u - S_d}
$$

After solving for $\Delta$, the risk-free position is:

$$
B = \frac{C_u - \Delta S_u}{R}
$$

The option price today is the cost of the replicating portfolio:

$$
C_0 = \Delta S_0 + B
$$

This price does not use the real-world probability of the up state.

It only uses no-arbitrage logic.

In [3]:
# ============================================================
# Replicating portfolio for a one-period option
# ============================================================

@dataclass(frozen=True)
class ReplicatingPortfolio:
    """
    Replicating portfolio for a one-period European option.

    delta:
        Number of stock shares held.
    bond:
        Dollars invested in the risk-free asset at time 0.
        A negative value means borrowing.
    option_price:
        Cost of the replicating portfolio at time 0.
    """

    delta: float
    bond: float
    option_price: float


def solve_replicating_portfolio(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
) -> ReplicatingPortfolio:
    """
    Solve for the one-period replicating portfolio.

    The portfolio is:

        delta shares of stock
        bond dollars in the risk-free asset today

    It must satisfy:

        delta * S_u + R * bond = payoff_up
        delta * S_d + R * bond = payoff_down
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)

    s_u = market.stock_up
    s_d = market.stock_down
    R = market.gross_risk_free_return

    payoff_up = european_option_payoff(s_u, strike, option_type)
    payoff_down = european_option_payoff(s_d, strike, option_type)

    delta = (payoff_up - payoff_down) / (s_u - s_d)
    bond = (payoff_up - delta * s_u) / R
    option_price = delta * market.s0 + bond

    return ReplicatingPortfolio(
        delta=delta,
        bond=bond,
        option_price=option_price,
    )


def replication_audit_table(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
) -> pd.DataFrame:
    """
    Show that the replicating portfolio matches the option payoff in both states.
    """
    portfolio = solve_replicating_portfolio(market, strike, option_type)
    R = market.gross_risk_free_return

    rows = []

    for state, stock_price in [
        ("up", market.stock_up),
        ("down", market.stock_down),
    ]:
        option_payoff = european_option_payoff(stock_price, strike, option_type)
        portfolio_value = portfolio.delta * stock_price + R * portfolio.bond
        replication_error = portfolio_value - option_payoff

        rows.append(
            {
                "state": state,
                "stock_price": stock_price,
                "option_payoff": option_payoff,
                "portfolio_value": portfolio_value,
                "replication_error": replication_error,
            }
        )

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Baseline replication result
# ------------------------------------------------------------

portfolio = solve_replicating_portfolio(
    market=market,
    strike=strike,
    option_type=option_type,
)

summary = pd.DataFrame(
    {
        "quantity": ["delta", "bond", "option_price"],
        "value": [portfolio.delta, portfolio.bond, portfolio.option_price],
    }
)

display(summary)
display(replication_audit_table(market, strike, option_type))

assert abs(replication_audit_table(market, strike, option_type)["replication_error"]).max() < TOL

print("Replication check passed.")

,quantity,value
0,delta,0.500000
1,bond,-38.095238
2,option_price,11.904762


,state,stock_price,option_payoff,portfolio_value,replication_error
0,up,120.000000,20.000000,20.000000,0.000000
1,down,80.000000,0.000000,0.000000,0.000000


Replication check passed.


## 3. Risk-neutral probability

The same option price can also be written using a special probability called the **risk-neutral probability**.

This is not the real-world probability of the stock going up.

It is the probability that makes the discounted stock price fair under the risk-free rate.

The one-period risk-neutral probability is:

$$
q = \frac{R - d}{u - d}
$$

where:

$$
R = 1 + r
$$

The no-arbitrage condition:

$$
d < R < u
$$

guarantees that:

$$
0 < q < 1
$$

Using this probability, the option price is the discounted expected payoff:

$$
C_0 = \frac{1}{R}\left(qC_u + (1-q)C_d\right)
$$

For a put option:

$$
P_0 = \frac{1}{R}\left(qP_u + (1-q)P_d\right)
$$

This formula gives the same price as the replicating portfolio.

That is the point: the risk-neutral probability is not introduced to forecast the stock. It is just another way to express the no-arbitrage replication price.

In [4]:
# ============================================================
# Risk-neutral probability and pricing
# ============================================================

def risk_neutral_probability(market: OnePeriodBinomialMarket) -> float:
    """
    Compute the one-period risk-neutral probability.

    q = (R - d) / (u - d)

    This is valid only under the no-arbitrage condition:

        d < R < u
    """
    market.validate_no_arbitrage()

    R = market.gross_risk_free_return
    q = (R - market.d) / (market.u - market.d)

    assert_between("risk_neutral_probability", q, 0.0, 1.0)

    return q


def price_by_risk_neutral_expectation(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
) -> float:
    """
    Price a one-period European option using the risk-neutral expectation.

    price = (q * payoff_up + (1 - q) * payoff_down) / R
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)

    q = risk_neutral_probability(market)
    R = market.gross_risk_free_return

    payoff_up = european_option_payoff(market.stock_up, strike, option_type)
    payoff_down = european_option_payoff(market.stock_down, strike, option_type)

    return (q * payoff_up + (1.0 - q) * payoff_down) / R


def pricing_method_comparison(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
) -> pd.DataFrame:
    """
    Compare the replication price and the risk-neutral expectation price.
    """
    portfolio = solve_replicating_portfolio(market, strike, option_type)
    rn_price = price_by_risk_neutral_expectation(market, strike, option_type)

    return pd.DataFrame(
        {
            "method": [
                "replicating_portfolio",
                "risk_neutral_expectation",
            ],
            "option_price": [
                portfolio.option_price,
                rn_price,
            ],
            "absolute_difference_vs_replication": [
                0.0,
                abs(rn_price - portfolio.option_price),
            ],
        }
    )


q = risk_neutral_probability(market)
rn_price = price_by_risk_neutral_expectation(market, strike, option_type)

summary = pd.DataFrame(
    {
        "quantity": [
            "gross_risk_free_return",
            "risk_neutral_probability",
            "risk_neutral_down_probability",
            "risk_neutral_price",
            "replication_price",
        ],
        "value": [
            market.gross_risk_free_return,
            q,
            1.0 - q,
            rn_price,
            portfolio.option_price,
        ],
    }
)

comparison = pricing_method_comparison(market, strike, option_type)

display(summary)
display(comparison)

assert abs(rn_price - portfolio.option_price) < TOL

print("Risk-neutral pricing check passed.")

,quantity,value
0,gross_risk_free_return,1.050000
1,risk_neutral_probability,0.625000
2,risk_neutral_down_probability,0.375000
3,risk_neutral_price,11.904762
4,replication_price,11.904762


,method,option_price,absolute_difference_vs_replication
0,replicating_portfolio,11.904762,0.000000
1,risk_neutral_expectation,11.904762,0.000000


Risk-neutral pricing check passed.


## 4. Real-world probability is not the pricing input

Now we separate two ideas:

1. **Real-world probability**  
   This is the trader's or analyst's belief about how likely the up state is.

2. **Risk-neutral probability**  
   This is the probability implied by no-arbitrage pricing.

The real-world probability can be useful for forecasting, expected returns, or trading views.

But it is not needed to price a derivative in a complete one-period binomial model.

Suppose the real-world up probability is:

$$
p = \mathbb{P}(\text{up})
$$

The real-world discounted expected payoff of a call would be:

$$
\frac{1}{R}\left(pC_u + (1-p)C_d\right)
$$

But this generally does **not** equal the no-arbitrage option price unless:

$$
p = q
$$

The no-arbitrage price uses:

$$
q = \frac{R-d}{u-d}
$$

not the real-world probability $p$.

This is the first major conceptual point of derivative pricing:

> The option is priced by replication, not by forecasting the stock's real-world direction.

In [5]:
# ============================================================
# Real-world probability versus risk-neutral probability
# ============================================================

def discounted_expected_payoff_under_real_probability(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
    real_world_up_probability: float,
) -> float:
    """
    Compute the discounted expected payoff using a chosen real-world probability.

    This is not generally the no-arbitrage price.

    It equals the no-arbitrage price only when:

        real_world_up_probability == risk_neutral_probability
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)
    assert_between(
        "real_world_up_probability",
        real_world_up_probability,
        0.0,
        1.0,
    )

    p = real_world_up_probability
    R = market.gross_risk_free_return

    payoff_up = european_option_payoff(market.stock_up, strike, option_type)
    payoff_down = european_option_payoff(market.stock_down, strike, option_type)

    return (p * payoff_up + (1.0 - p) * payoff_down) / R


def real_vs_risk_neutral_probability_audit(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
    real_world_probabilities: list[float],
) -> pd.DataFrame:
    """
    Compare discounted expected payoff under real-world probabilities
    against the no-arbitrage replication price.
    """
    replication_price = solve_replicating_portfolio(
        market,
        strike,
        option_type,
    ).option_price

    q = risk_neutral_probability(market)

    rows = []

    for p in real_world_probabilities:
        expected_payoff_price = discounted_expected_payoff_under_real_probability(
            market=market,
            strike=strike,
            option_type=option_type,
            real_world_up_probability=p,
        )

        rows.append(
            {
                "real_world_up_probability": p,
                "risk_neutral_probability": q,
                "discounted_expected_payoff_price": expected_payoff_price,
                "no_arbitrage_replication_price": replication_price,
                "pricing_difference": expected_payoff_price - replication_price,
                "matches_no_arbitrage_price": abs(expected_payoff_price - replication_price) < TOL,
            }
        )

    return pd.DataFrame(rows)


real_world_probabilities = [
    0.20,
    0.40,
    risk_neutral_probability(market),
    0.80,
]

real_probability_audit = real_vs_risk_neutral_probability_audit(
    market=market,
    strike=strike,
    option_type=option_type,
    real_world_probabilities=real_world_probabilities,
)

display(real_probability_audit)

assert real_probability_audit["matches_no_arbitrage_price"].sum() == 1

print("Real-world probability separation check passed.")

,real_world_up_probability,risk_neutral_probability,discounted_expected_payoff_price,no_arbitrage_replication_price,pricing_difference,matches_no_arbitrage_price
0,0.200000,0.625000,3.809524,11.904762,-8.095238,False
1,0.400000,0.625000,7.619048,11.904762,-4.285714,False
2,0.625000,0.625000,11.904762,11.904762,0.000000,True
3,0.800000,0.625000,15.238095,11.904762,3.333333,False


Real-world probability separation check passed.


## 5. What happens if the market option price is wrong?

The replication price is the only no-arbitrage price.

Suppose the market quotes the option at price $M_0$.

The no-arbitrage replication price is:

$$
C_0 = \Delta S_0 + B
$$

If the market price is higher than the replication price:

$$
M_0 > C_0
$$

then the option is overpriced relative to the replicating portfolio.

An arbitrage strategy is:

1. Sell the overpriced option.
2. Buy the cheaper replicating portfolio.
3. Keep the difference as riskless profit.

The future payoff is zero in every state because the replicating portfolio exactly covers the option liability.

If the market price is lower than the replication price:

$$
M_0 < C_0
$$

then the option is underpriced relative to the replicating portfolio.

An arbitrage strategy is:

1. Buy the underpriced option.
2. Short the expensive replicating portfolio.
3. Keep the difference as riskless profit.

Again, the future payoff is zero in every state because the option and the portfolio have identical future payoffs.

So the pricing rule is:

$$
M_0 = C_0
$$

Any other price creates arbitrage.

In [6]:
# ============================================================
# Arbitrage audit when the quoted market option price is wrong
# ============================================================

def arbitrage_strategy_from_market_price(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
    market_option_price: float,
) -> dict[str, object]:
    """
    Determine the arbitrage strategy implied by a mispriced option.

    If market price > replication price:
        sell option, buy replicating portfolio

    If market price < replication price:
        buy option, short replicating portfolio

    If market price == replication price:
        no arbitrage.
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)
    assert_nonnegative("market_option_price", market_option_price)

    portfolio = solve_replicating_portfolio(market, strike, option_type)
    fair_price = portfolio.option_price
    price_gap = market_option_price - fair_price

    if abs(price_gap) < TOL:
        return {
            "market_option_price": market_option_price,
            "fair_replication_price": fair_price,
            "price_gap": price_gap,
            "strategy": "no arbitrage",
            "initial_cash_profit": 0.0,
            "position_option": 0.0,
            "position_replicating_portfolio": 0.0,
        }

    if price_gap > 0:
        return {
            "market_option_price": market_option_price,
            "fair_replication_price": fair_price,
            "price_gap": price_gap,
            "strategy": "sell overpriced option, buy replicating portfolio",
            "initial_cash_profit": price_gap,
            "position_option": -1.0,
            "position_replicating_portfolio": 1.0,
        }

    return {
        "market_option_price": market_option_price,
        "fair_replication_price": fair_price,
        "price_gap": price_gap,
        "strategy": "buy underpriced option, short replicating portfolio",
        "initial_cash_profit": -price_gap,
        "position_option": 1.0,
        "position_replicating_portfolio": -1.0,
    }


def arbitrage_terminal_payoff_table(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
    market_option_price: float,
) -> pd.DataFrame:
    """
    Show terminal payoff of the arbitrage strategy in each state.
    """
    strategy = arbitrage_strategy_from_market_price(
        market=market,
        strike=strike,
        option_type=option_type,
        market_option_price=market_option_price,
    )

    portfolio = solve_replicating_portfolio(market, strike, option_type)
    R = market.gross_risk_free_return

    rows = []

    for state, stock_price in [
        ("up", market.stock_up),
        ("down", market.stock_down),
    ]:
        option_payoff = european_option_payoff(stock_price, strike, option_type)
        replicating_portfolio_payoff = portfolio.delta * stock_price + R * portfolio.bond

        terminal_net_payoff = (
            strategy["position_option"] * option_payoff
            + strategy["position_replicating_portfolio"] * replicating_portfolio_payoff
        )

        rows.append(
            {
                "state": state,
                "stock_price": stock_price,
                "option_payoff": option_payoff,
                "replicating_portfolio_payoff": replicating_portfolio_payoff,
                "position_option": strategy["position_option"],
                "position_replicating_portfolio": strategy["position_replicating_portfolio"],
                "terminal_net_payoff": terminal_net_payoff,
            }
        )

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Test three quoted market prices:
# 1. underpriced
# 2. fair
# 3. overpriced
# ------------------------------------------------------------

fair_price = portfolio.option_price

market_prices = [
    fair_price - 2.00,
    fair_price,
    fair_price + 2.00,
]

strategy_rows = [
    arbitrage_strategy_from_market_price(
        market=market,
        strike=strike,
        option_type=option_type,
        market_option_price=market_price,
    )
    for market_price in market_prices
]

strategy_audit = pd.DataFrame(strategy_rows)

display(strategy_audit)

for market_price in market_prices:
    print(f"\nMarket option price: {market_price:.6f}")
    display(
        arbitrage_terminal_payoff_table(
            market=market,
            strike=strike,
            option_type=option_type,
            market_option_price=market_price,
        )
    )

print("Arbitrage audit complete.")

,market_option_price,fair_replication_price,price_gap,strategy,initial_cash_profit,position_option,position_replicating_portfolio
0,9.904762,11.904762,-2.000000,"buy underpriced option, short replicating port...",2.000000,1.000000,-1.000000
1,11.904762,11.904762,0.000000,no arbitrage,0.000000,0.000000,0.000000
2,13.904762,11.904762,2.000000,"sell overpriced option, buy replicating portfolio",2.000000,-1.000000,1.000000



Market option price: 9.904762


,state,stock_price,option_payoff,replicating_portfolio_payoff,position_option,position_replicating_portfolio,terminal_net_payoff
0,up,120.000000,20.000000,20.000000,1.000000,-1.000000,0.000000
1,down,80.000000,0.000000,0.000000,1.000000,-1.000000,0.000000



Market option price: 11.904762


,state,stock_price,option_payoff,replicating_portfolio_payoff,position_option,position_replicating_portfolio,terminal_net_payoff
0,up,120.000000,20.000000,20.000000,0.000000,0.000000,0.000000
1,down,80.000000,0.000000,0.000000,0.000000,0.000000,0.000000



Market option price: 13.904762


,state,stock_price,option_payoff,replicating_portfolio_payoff,position_option,position_replicating_portfolio,terminal_net_payoff
0,up,120.000000,20.000000,20.000000,-1.000000,1.000000,0.000000
1,down,80.000000,0.000000,0.000000,-1.000000,1.000000,0.000000


Arbitrage audit complete.


## 6. Put-call parity in the one-period model

The same no-arbitrage logic also gives **put-call parity**.

For a European call and European put with the same:

- underlying stock
- strike $K$
- maturity
- risk-free rate

the following relationship must hold:

$$
C_0 - P_0 = S_0 - \frac{K}{R}
$$

Equivalently:

$$
C_0 + \frac{K}{R} = P_0 + S_0
$$

The left side is:

- long one call
- enough risk-free money today to pay $K$ at maturity

The right side is:

- long one put
- long one share of stock

At maturity, both portfolios have the same payoff.

If the stock ends above the strike:

$$
S_T > K
$$

then:

$$
\text{call} + K = (S_T - K) + K = S_T
$$

and:

$$
\text{put} + S_T = 0 + S_T = S_T
$$

If the stock ends below the strike:

$$
S_T < K
$$

then:

$$
\text{call} + K = 0 + K = K
$$

and:

$$
\text{put} + S_T = (K - S_T) + S_T = K
$$

So both portfolios always pay:

$$
\max(S_T, K)
$$

Because they have the same future payoff in every state, they must have the same price today.

That is put-call parity.

In [7]:
# ============================================================
# Put-call parity check in the one-period model
# ============================================================

def put_call_parity_audit(
    market: OnePeriodBinomialMarket,
    strike: float,
) -> pd.DataFrame:
    """
    Check one-period European put-call parity.

    Put-call parity:

        C0 - P0 = S0 - K / R

    Equivalent form:

        C0 + K / R = P0 + S0
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)

    R = market.gross_risk_free_return

    call_price = solve_replicating_portfolio(
        market=market,
        strike=strike,
        option_type="call",
    ).option_price

    put_price = solve_replicating_portfolio(
        market=market,
        strike=strike,
        option_type="put",
    ).option_price

    lhs_difference_form = call_price - put_price
    rhs_difference_form = market.s0 - strike / R

    lhs_portfolio_form = call_price + strike / R
    rhs_portfolio_form = put_price + market.s0

    return pd.DataFrame(
        {
            "check": [
                "C0",
                "P0",
                "C0 - P0",
                "S0 - K/R",
                "difference_form_error",
                "C0 + K/R",
                "P0 + S0",
                "portfolio_form_error",
            ],
            "value": [
                call_price,
                put_price,
                lhs_difference_form,
                rhs_difference_form,
                lhs_difference_form - rhs_difference_form,
                lhs_portfolio_form,
                rhs_portfolio_form,
                lhs_portfolio_form - rhs_portfolio_form,
            ],
        }
    )


def put_call_parity_terminal_payoff_table(
    market: OnePeriodBinomialMarket,
    strike: float,
) -> pd.DataFrame:
    """
    Show that the two put-call parity portfolios have identical payoffs.

    Portfolio A:
        long call + risk-free bond that pays K at maturity

    Portfolio B:
        long put + one share of stock
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)

    rows = []

    for state, stock_price in [
        ("up", market.stock_up),
        ("down", market.stock_down),
    ]:
        call_payoff = european_option_payoff(stock_price, strike, "call")
        put_payoff = european_option_payoff(stock_price, strike, "put")

        call_plus_bond_payoff = call_payoff + strike
        put_plus_stock_payoff = put_payoff + stock_price

        rows.append(
            {
                "state": state,
                "stock_price": stock_price,
                "call_payoff": call_payoff,
                "put_payoff": put_payoff,
                "call_plus_bond_payoff": call_plus_bond_payoff,
                "put_plus_stock_payoff": put_plus_stock_payoff,
                "payoff_difference": call_plus_bond_payoff - put_plus_stock_payoff,
            }
        )

    return pd.DataFrame(rows)


parity_audit = put_call_parity_audit(
    market=market,
    strike=strike,
)

parity_payoff_audit = put_call_parity_terminal_payoff_table(
    market=market,
    strike=strike,
)

display(parity_audit)
display(parity_payoff_audit)

max_parity_error = parity_audit.loc[
    parity_audit["check"].isin(["difference_form_error", "portfolio_form_error"]),
    "value",
].abs().max()

max_payoff_error = parity_payoff_audit["payoff_difference"].abs().max()

assert max_parity_error < TOL
assert max_payoff_error < TOL

print("Put-call parity check passed.")

,check,value
0,C0,11.904762
1,P0,7.142857
2,C0 - P0,4.761905
3,S0 - K/R,4.761905
4,difference_form_error,0.000000
5,C0 + K/R,107.142857
6,P0 + S0,107.142857
7,portfolio_form_error,0.000000


,state,stock_price,call_payoff,put_payoff,call_plus_bond_payoff,put_plus_stock_payoff,payoff_difference
0,up,120.000000,20.000000,0.000000,120.000000,120.000000,0.000000
1,down,80.000000,0.000000,20.000000,100.000000,100.000000,0.000000


Put-call parity check passed.


## 7. Two-period binomial model

The one-period model can be extended by adding more time steps.

Now the stock moves twice.

At each step, it can move up by multiplier $u$ or down by multiplier $d$.

Starting from $S_0$, the possible terminal stock prices after two periods are:

$$
S_{uu} = u^2 S_0
$$

$$
S_{ud} = ud S_0
$$

$$
S_{dd} = d^2 S_0
$$

The middle node can be reached two ways:

$$
S_{ud} = S_{du}
$$

This is called a **recombining tree**.

The key idea is backward induction:

1. Compute the option payoff at maturity.
2. Move one step backward and price each node using the one-period risk-neutral formula.
3. Move backward again to get the option price at time 0.

At each node, the one-step pricing formula is:

$$
V = \frac{1}{R}\left(qV_u + (1-q)V_d\right)
$$

where:

$$
q = \frac{R-d}{u-d}
$$

This is the same no-arbitrage logic as before, applied repeatedly.

The two-period model is not a new pricing idea. It is the one-period replication argument applied node by node.

In [8]:
# ============================================================
# Two-period binomial pricing by backward induction
# ============================================================

@dataclass(frozen=True)
class TwoPeriodBinomialPrice:
    """
    Two-period binomial option value tree.

    The terminal nodes are:
        uu, ud, dd

    The one-step-back nodes are:
        u, d

    The initial node is:
        0
    """

    value_0: float
    value_u: float
    value_d: float
    payoff_uu: float
    payoff_ud: float
    payoff_dd: float


def two_period_stock_tree(market: OnePeriodBinomialMarket) -> pd.DataFrame:
    """
    Return the recombining two-period stock tree.
    """
    market.validate_no_arbitrage()

    rows = [
        {
            "time": 0,
            "node": "0",
            "path": "",
            "up_moves": 0,
            "down_moves": 0,
            "stock_price": market.s0,
        },
        {
            "time": 1,
            "node": "u",
            "path": "u",
            "up_moves": 1,
            "down_moves": 0,
            "stock_price": market.s0 * market.u,
        },
        {
            "time": 1,
            "node": "d",
            "path": "d",
            "up_moves": 0,
            "down_moves": 1,
            "stock_price": market.s0 * market.d,
        },
        {
            "time": 2,
            "node": "uu",
            "path": "uu",
            "up_moves": 2,
            "down_moves": 0,
            "stock_price": market.s0 * market.u**2,
        },
        {
            "time": 2,
            "node": "ud",
            "path": "ud / du",
            "up_moves": 1,
            "down_moves": 1,
            "stock_price": market.s0 * market.u * market.d,
        },
        {
            "time": 2,
            "node": "dd",
            "path": "dd",
            "up_moves": 0,
            "down_moves": 2,
            "stock_price": market.s0 * market.d**2,
        },
    ]

    return pd.DataFrame(rows)


def price_two_period_option_by_backward_induction(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
) -> TwoPeriodBinomialPrice:
    """
    Price a two-period European option by backward induction.

    Step 1:
        Compute terminal payoffs at uu, ud, dd.

    Step 2:
        Price the time-1 up and down nodes using the one-period
        risk-neutral pricing rule.

    Step 3:
        Price the time-0 node using the same one-period rule.
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)

    q = risk_neutral_probability(market)
    R = market.gross_risk_free_return

    s_uu = market.s0 * market.u**2
    s_ud = market.s0 * market.u * market.d
    s_dd = market.s0 * market.d**2

    payoff_uu = european_option_payoff(s_uu, strike, option_type)
    payoff_ud = european_option_payoff(s_ud, strike, option_type)
    payoff_dd = european_option_payoff(s_dd, strike, option_type)

    value_u = (q * payoff_uu + (1.0 - q) * payoff_ud) / R
    value_d = (q * payoff_ud + (1.0 - q) * payoff_dd) / R

    value_0 = (q * value_u + (1.0 - q) * value_d) / R

    return TwoPeriodBinomialPrice(
        value_0=value_0,
        value_u=value_u,
        value_d=value_d,
        payoff_uu=payoff_uu,
        payoff_ud=payoff_ud,
        payoff_dd=payoff_dd,
    )


def two_period_value_tree_table(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
) -> pd.DataFrame:
    """
    Return stock prices and option values across the two-period tree.
    """
    price = price_two_period_option_by_backward_induction(
        market=market,
        strike=strike,
        option_type=option_type,
    )

    stock_tree = two_period_stock_tree(market)

    node_to_option_value = {
        "0": price.value_0,
        "u": price.value_u,
        "d": price.value_d,
        "uu": price.payoff_uu,
        "ud": price.payoff_ud,
        "dd": price.payoff_dd,
    }

    stock_tree["option_value"] = stock_tree["node"].map(node_to_option_value)
    stock_tree["is_terminal_payoff"] = stock_tree["time"].eq(2)

    return stock_tree


def price_two_period_option_by_terminal_expectation(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
) -> float:
    """
    Price a two-period European option directly from terminal payoffs.

    This is equivalent to backward induction in a recombining binomial tree.

    price = E_Q[payoff] / R^2
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)

    q = risk_neutral_probability(market)
    R = market.gross_risk_free_return

    s_uu = market.s0 * market.u**2
    s_ud = market.s0 * market.u * market.d
    s_dd = market.s0 * market.d**2

    payoff_uu = european_option_payoff(s_uu, strike, option_type)
    payoff_ud = european_option_payoff(s_ud, strike, option_type)
    payoff_dd = european_option_payoff(s_dd, strike, option_type)

    expected_payoff = (
        q**2 * payoff_uu
        + 2.0 * q * (1.0 - q) * payoff_ud
        + (1.0 - q) ** 2 * payoff_dd
    )

    return expected_payoff / R**2


# ------------------------------------------------------------
# Baseline two-period result
# ------------------------------------------------------------

two_period_price = price_two_period_option_by_backward_induction(
    market=market,
    strike=strike,
    option_type=option_type,
)

two_period_expectation_price = price_two_period_option_by_terminal_expectation(
    market=market,
    strike=strike,
    option_type=option_type,
)

two_period_comparison = pd.DataFrame(
    {
        "method": [
            "backward_induction",
            "terminal_risk_neutral_expectation",
        ],
        "option_price": [
            two_period_price.value_0,
            two_period_expectation_price,
        ],
        "absolute_difference_vs_backward_induction": [
            0.0,
            abs(two_period_expectation_price - two_period_price.value_0),
        ],
    }
)

display(two_period_value_tree_table(market, strike, option_type))
display(two_period_comparison)

assert abs(two_period_expectation_price - two_period_price.value_0) < TOL

print("Two-period backward induction check passed.")

,time,node,path,up_moves,down_moves,stock_price,option_value,is_terminal_payoff
0,0,0,,0,0,100.000000,15.589569,False
1,1,u,u,1,0,120.000000,26.190476,False
2,1,d,d,0,1,80.000000,0.000000,False
3,2,uu,uu,2,0,144.000000,44.000000,True
4,2,ud,ud / du,1,1,96.000000,0.000000,True
5,2,dd,dd,0,2,64.000000,0.000000,True


,method,option_price,absolute_difference_vs_backward_induction
0,backward_induction,15.589569,0.000000
1,terminal_risk_neutral_expectation,15.589569,0.000000


Two-period backward induction check passed.


## 8. General $N$-period binomial model

The two-period model can be generalized to $N$ periods.

At each period, the stock either moves up by $u$ or down by $d$.

After $N$ periods, a terminal node with $j$ up moves has stock price:

$$
S_{N,j} = S_0 u^j d^{N-j}
$$

where:

- $N$ is the number of time steps
- $j$ is the number of up moves
- $N-j$ is the number of down moves

The terminal option payoff is:

$$
V_{N,j} = \max(S_{N,j} - K, 0)
$$

for a call, and:

$$
V_{N,j} = \max(K - S_{N,j}, 0)
$$

for a put.

Then we work backward through the tree.

At any earlier node:

$$
V_{n,j} = \frac{1}{R}\left(qV_{n+1,j+1} + (1-q)V_{n+1,j}\right)
$$

where:

$$
q = \frac{R-d}{u-d}
$$

This is just the one-period pricing formula repeated node by node.

As $N$ becomes large and the time step becomes small, the binomial model begins to approximate continuous-time option pricing.

This is the bridge toward Black-Scholes.

But the conceptual foundation does not change:

> Price by no-arbitrage backward induction, not by real-world forecasting.

In [9]:
# ============================================================
# General N-period binomial pricing by backward induction
# ============================================================

@dataclass(frozen=True)
class NPeriodBinomialPrice:
    """
    N-period binomial option-pricing result.

    price:
        Option value at time 0.
    n_periods:
        Number of binomial time steps.
    risk_neutral_probability:
        One-step risk-neutral up probability.
    stock_layers:
        Stock prices at each time step.
    value_layers:
        Option values at each time step.
    """

    price: float
    n_periods: int
    risk_neutral_probability: float
    stock_layers: dict[int, np.ndarray]
    value_layers: dict[int, np.ndarray]


def assert_positive_integer(name: str, value: int) -> None:
    """Raise an error if value is not a positive integer."""
    if not isinstance(value, int) or value <= 0:
        raise ValueError(f"{name} must be a positive integer. Got {value!r}.")


def stock_prices_at_step(
    market: OnePeriodBinomialMarket,
    step: int,
) -> np.ndarray:
    """
    Return stock prices at a given binomial step.

    At step n, node j has j up moves and n-j down moves:

        S(n, j) = S0 * u^j * d^(n-j)

    The returned array is ordered by j = 0, 1, ..., n.
    """
    market.validate_no_arbitrage()

    if step < 0:
        raise ValueError(f"step must be nonnegative. Got {step!r}.")

    up_moves = np.arange(step + 1)
    down_moves = step - up_moves

    return market.s0 * (market.u ** up_moves) * (market.d ** down_moves)


def price_n_period_option_by_backward_induction(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
    n_periods: int,
) -> NPeriodBinomialPrice:
    """
    Price an N-period European option by backward induction.

    This applies the one-period no-arbitrage pricing rule repeatedly:

        V(n, j) = [q * V(n+1, j+1) + (1-q) * V(n+1, j)] / R

    where j is the number of up moves at node (n, j).
    """
    market.validate_no_arbitrage()
    assert_positive("strike", strike)
    assert_positive_integer("n_periods", n_periods)

    q = risk_neutral_probability(market)
    R = market.gross_risk_free_return

    stock_layers: dict[int, np.ndarray] = {
        step: stock_prices_at_step(market, step)
        for step in range(n_periods + 1)
    }

    terminal_stock_prices = stock_layers[n_periods]

    terminal_values = np.array(
        [
            european_option_payoff(stock_price, strike, option_type)
            for stock_price in terminal_stock_prices
        ],
        dtype=float,
    )

    value_layers: dict[int, np.ndarray] = {
        n_periods: terminal_values
    }

    current_values = terminal_values

    for step in range(n_periods - 1, -1, -1):
        current_values = (
            q * current_values[1:]
            + (1.0 - q) * current_values[:-1]
        ) / R

        value_layers[step] = current_values

    price = float(value_layers[0][0])

    return NPeriodBinomialPrice(
        price=price,
        n_periods=n_periods,
        risk_neutral_probability=q,
        stock_layers=stock_layers,
        value_layers=value_layers,
    )


def n_period_tree_table(
    result: NPeriodBinomialPrice,
) -> pd.DataFrame:
    """
    Convert an N-period binomial result into a reader-facing node table.
    """
    rows = []

    for step in range(result.n_periods + 1):
        stock_values = result.stock_layers[step]
        option_values = result.value_layers[step]

        for up_moves, (stock_price, option_value) in enumerate(
            zip(stock_values, option_values)
        ):
            rows.append(
                {
                    "time_step": step,
                    "node": f"({step}, {up_moves})",
                    "up_moves": up_moves,
                    "down_moves": step - up_moves,
                    "stock_price": stock_price,
                    "option_value": option_value,
                    "is_terminal": step == result.n_periods,
                }
            )

    return pd.DataFrame(rows)


def n_period_price_audit(
    market: OnePeriodBinomialMarket,
    strike: float,
    option_type: OptionType,
    n_values: list[int],
) -> pd.DataFrame:
    """
    Price the same option across several tree depths.

    Note:
    This does not yet use Black-Scholes time scaling.
    It simply applies the same per-step u, d, and r repeatedly.
    """
    rows = []

    for n_periods in n_values:
        result = price_n_period_option_by_backward_induction(
            market=market,
            strike=strike,
            option_type=option_type,
            n_periods=n_periods,
        )

        rows.append(
            {
                "n_periods": n_periods,
                "option_price": result.price,
                "risk_neutral_probability": result.risk_neutral_probability,
                "terminal_node_count": n_periods + 1,
            }
        )

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Consistency checks against previous one-period and two-period results
# ------------------------------------------------------------

one_period_general = price_n_period_option_by_backward_induction(
    market=market,
    strike=strike,
    option_type=option_type,
    n_periods=1,
)

two_period_general = price_n_period_option_by_backward_induction(
    market=market,
    strike=strike,
    option_type=option_type,
    n_periods=2,
)

assert abs(one_period_general.price - portfolio.option_price) < TOL
assert abs(two_period_general.price - two_period_price.value_0) < TOL


# ------------------------------------------------------------
# Display a small N-period tree and a depth audit
# ------------------------------------------------------------

three_period_result = price_n_period_option_by_backward_induction(
    market=market,
    strike=strike,
    option_type=option_type,
    n_periods=3,
)

display(n_period_tree_table(three_period_result))

display(
    n_period_price_audit(
        market=market,
        strike=strike,
        option_type=option_type,
        n_values=[1, 2, 3, 4, 5],
    )
)

print("General N-period backward induction check passed.")

,time_step,node,up_moves,down_moves,stock_price,option_value,is_terminal
0,0,"(0, 0)",0,0,100.000000,21.123529,False
1,1,"(1, 0)",0,1,80.000000,5.385488,False
2,1,"(1, 1)",1,0,120.000000,32.256236,False
3,2,"(2, 0)",0,2,64.000000,0.000000,False
4,2,"(2, 1)",1,1,96.000000,9.047619,False
5,2,"(2, 2)",2,0,144.000000,48.761905,False
6,3,"(3, 0)",0,3,51.200000,0.000000,True
7,3,"(3, 1)",1,2,76.800000,0.000000,True
8,3,"(3, 2)",2,1,115.200000,15.200000,True
9,3,"(3, 3)",3,0,172.800000,72.800000,True


,n_periods,option_price,risk_neutral_probability,terminal_node_count
0,1,11.904762,0.625000,2
1,2,15.589569,0.625000,3
2,3,21.123529,0.625000,4
3,4,24.998425,0.625000,5
4,5,28.740451,0.625000,6


General N-period backward induction check passed.


## 9. Time scaling and convergence intuition

The previous $N$-period example repeated the same $u$, $d$, and $r$ at every step.

That is useful for understanding backward induction, but it is not yet the correct way to study convergence toward continuous-time pricing.

To approximate a fixed maturity $T$, the time step must shrink as the number of periods increases:

$$
\Delta t = \frac{T}{N}
$$

The risk-free gross return per step becomes:

$$
R_{\Delta t} = e^{r \Delta t}
$$

A common binomial scaling is the Cox-Ross-Rubinstein structure:

$$
u = e^{\sigma \sqrt{\Delta t}}
$$

$$
d = e^{-\sigma \sqrt{\Delta t}}
$$

where:

- $\sigma$ is volatility
- $T$ is maturity in years
- $N$ is the number of binomial steps
- $\Delta t$ is the length of each step

The risk-neutral probability becomes:

$$
q = \frac{R_{\Delta t} - d}{u - d}
$$

As $N$ increases:

- each individual step becomes smaller
- the tree has more possible terminal prices
- the binomial distribution becomes smoother
- the model moves closer to continuous-time option pricing

This is the bridge from discrete no-arbitrage replication to the Black-Scholes model.

The next code cell builds a time-scaled binomial pricer and checks how the price changes as $N$ increases.

In [10]:
# ============================================================
# Time-scaled Cox-Ross-Rubinstein binomial pricing
# ============================================================

@dataclass(frozen=True)
class CRRStepParameters:
    """
    Cox-Ross-Rubinstein step parameters.

    dt:
        Length of each time step.
    u:
        Up multiplier per step.
    d:
        Down multiplier per step.
    R:
        Gross risk-free return per step.
    q:
        Risk-neutral up probability per step.
    """

    dt: float
    u: float
    d: float
    R: float
    q: float


def standard_normal_cdf(x: float) -> float:
    """
    Standard normal cumulative distribution function.

    This avoids adding scipy as a dependency in this notebook.
    """
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def black_scholes_price_reference(
    s0: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    volatility: float,
    option_type: OptionType,
) -> float:
    """
    Black-Scholes European option price.

    This is used only as a numerical reference for the convergence check.
    The derivation is not part of this notebook.
    """
    assert_positive("s0", s0)
    assert_positive("strike", strike)
    assert_positive("maturity", maturity)
    assert_nonnegative("risk_free_rate", risk_free_rate)
    assert_positive("volatility", volatility)

    sqrt_T = math.sqrt(maturity)

    d1 = (
        math.log(s0 / strike)
        + (risk_free_rate + 0.5 * volatility**2) * maturity
    ) / (volatility * sqrt_T)

    d2 = d1 - volatility * sqrt_T

    discounted_strike = strike * math.exp(-risk_free_rate * maturity)

    if option_type == "call":
        return (
            s0 * standard_normal_cdf(d1)
            - discounted_strike * standard_normal_cdf(d2)
        )

    if option_type == "put":
        return (
            discounted_strike * standard_normal_cdf(-d2)
            - s0 * standard_normal_cdf(-d1)
        )

    raise ValueError(f"Unsupported option_type: {option_type!r}")


def crr_step_parameters(
    maturity: float,
    risk_free_rate: float,
    volatility: float,
    n_periods: int,
) -> CRRStepParameters:
    """
    Compute Cox-Ross-Rubinstein parameters for a fixed maturity.

    dt = T / N
    u = exp(sigma * sqrt(dt))
    d = exp(-sigma * sqrt(dt))
    R = exp(r * dt)
    q = (R - d) / (u - d)
    """
    assert_positive("maturity", maturity)
    assert_nonnegative("risk_free_rate", risk_free_rate)
    assert_positive("volatility", volatility)
    assert_positive_integer("n_periods", n_periods)

    dt = maturity / n_periods
    u = math.exp(volatility * math.sqrt(dt))
    d = math.exp(-volatility * math.sqrt(dt))
    R = math.exp(risk_free_rate * dt)

    if not d < R < u:
        raise ValueError(
            "CRR no-arbitrage condition failed. Expected d < R < u. "
            f"Got d={d:.6f}, R={R:.6f}, u={u:.6f}."
        )

    q = (R - d) / (u - d)

    assert_between("risk_neutral_probability", q, 0.0, 1.0)

    return CRRStepParameters(
        dt=dt,
        u=u,
        d=d,
        R=R,
        q=q,
    )


def price_crr_binomial_european_option(
    s0: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    volatility: float,
    option_type: OptionType,
    n_periods: int,
) -> float:
    """
    Price a European option with a time-scaled CRR binomial tree.
    """
    assert_positive("s0", s0)
    assert_positive("strike", strike)

    params = crr_step_parameters(
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        volatility=volatility,
        n_periods=n_periods,
    )

    up_moves = np.arange(n_periods + 1)
    down_moves = n_periods - up_moves

    terminal_stock_prices = (
        s0
        * params.u ** up_moves
        * params.d ** down_moves
    )

    terminal_values = np.array(
        [
            european_option_payoff(stock_price, strike, option_type)
            for stock_price in terminal_stock_prices
        ],
        dtype=float,
    )

    current_values = terminal_values

    for _ in range(n_periods):
        current_values = (
            params.q * current_values[1:]
            + (1.0 - params.q) * current_values[:-1]
        ) / params.R

    return float(current_values[0])


def crr_convergence_audit(
    s0: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    volatility: float,
    option_type: OptionType,
    n_values: list[int],
) -> pd.DataFrame:
    """
    Compare CRR binomial prices across increasing tree depths.
    """
    reference_price = black_scholes_price_reference(
        s0=s0,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        volatility=volatility,
        option_type=option_type,
    )

    rows = []

    previous_price = None

    for n_periods in n_values:
        params = crr_step_parameters(
            maturity=maturity,
            risk_free_rate=risk_free_rate,
            volatility=volatility,
            n_periods=n_periods,
        )

        binomial_price = price_crr_binomial_european_option(
            s0=s0,
            strike=strike,
            maturity=maturity,
            risk_free_rate=risk_free_rate,
            volatility=volatility,
            option_type=option_type,
            n_periods=n_periods,
        )

        rows.append(
            {
                "n_periods": n_periods,
                "dt": params.dt,
                "u": params.u,
                "d": params.d,
                "R": params.R,
                "q": params.q,
                "crr_binomial_price": binomial_price,
                "black_scholes_reference": reference_price,
                "error_vs_reference": binomial_price - reference_price,
                "abs_error_vs_reference": abs(binomial_price - reference_price),
                "change_vs_previous_n": (
                    np.nan if previous_price is None else binomial_price - previous_price
                ),
            }
        )

        previous_price = binomial_price

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Time-scaled convergence example
# ------------------------------------------------------------

crr_s0 = 100.0
crr_strike = 100.0
crr_maturity = 1.0
crr_risk_free_rate = 0.05
crr_volatility = 0.20
crr_option_type: OptionType = "call"

n_values = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

convergence_audit = crr_convergence_audit(
    s0=crr_s0,
    strike=crr_strike,
    maturity=crr_maturity,
    risk_free_rate=crr_risk_free_rate,
    volatility=crr_volatility,
    option_type=crr_option_type,
    n_values=n_values,
)

display(convergence_audit)

assert convergence_audit["abs_error_vs_reference"].iloc[-1] < convergence_audit["abs_error_vs_reference"].iloc[0]

print("Time-scaled CRR convergence check passed.")

,n_periods,dt,u,d,R,q,crr_binomial_price,black_scholes_reference,error_vs_reference,abs_error_vs_reference,change_vs_previous_n
0,1,1.000000,1.221403,0.818731,1.051271,0.577493,12.162285,10.450584,1.711701,1.711701,NaN
1,2,0.500000,1.151910,0.868123,1.025315,0.553908,9.540501,10.450584,-0.910082,0.910082,-2.621784
2,4,0.250000,1.105171,0.904837,1.012578,0.537808,9.970523,10.450584,-0.480061,0.480061,0.430022
3,8,0.125000,1.073271,0.931731,1.006270,0.526625,10.205099,10.450584,-0.245484,0.245484,0.234576
4,16,0.062500,1.051271,0.951229,1.003130,0.518788,10.326651,10.450584,-0.123933,0.123933,0.121552
5,32,0.031250,1.035988,0.965262,1.001564,0.513272,10.388346,10.450584,-0.062238,0.062238,0.061695
6,64,0.015625,1.025315,0.975310,1.000782,0.509380,10.419400,10.450584,-0.031183,0.031183,0.031055
7,128,0.007812,1.017835,0.982478,1.000391,0.506631,10.434976,10.450584,-0.015607,0.015607,0.015576
8,256,0.003906,1.012578,0.987578,1.000195,0.504688,10.442776,10.450584,-0.007808,0.007808,0.007800
9,512,0.001953,1.008878,0.991200,1.000098,0.503315,10.446679,10.450584,-0.003905,0.003905,0.003903


Time-scaled CRR convergence check passed.


## 10. Interpreting the convergence check

The convergence table shows the role of time scaling.

When $N$ increases, the maturity $T$ stays fixed, but each step becomes smaller:

$$
\Delta t = \frac{T}{N}
$$

The up and down multipliers move closer to 1:

$$
u = e^{\sigma\sqrt{\Delta t}}
$$

$$
d = e^{-\sigma\sqrt{\Delta t}}
$$

The per-step risk-free return also moves closer to 1:

$$
R_{\Delta t} = e^{r\Delta t}
$$

In the output, the CRR binomial price moves toward the Black-Scholes reference price.

This does not mean Black-Scholes has been fully proven in this notebook.

It only shows the numerical bridge:

> A time-scaled no-arbitrage binomial tree approaches the continuous-time Black-Scholes price as the number of steps increases.

The important point is that the pricing logic remains the same:

1. Build a stock tree.
2. Compute terminal option payoffs.
3. Discount backward using the risk-neutral probability.
4. Obtain the time-0 no-arbitrage price.

This completes the first notebook's main task: derivative pricing from no-arbitrage replication rather than real-world forecasting.

In [11]:
# ============================================================
# Final notebook audit
# ============================================================

def final_notebook_audit() -> pd.DataFrame:
    """
    Summarize the core checks required for this notebook to pass.

    The notebook passes only if pricing is grounded in:
        - no-arbitrage
        - replication
        - risk-neutral valuation as equivalent expression
        - backward induction

    It does not pass by using real-world probability as the pricing input.
    """
    one_period_replication_error = replication_audit_table(
        market=market,
        strike=strike,
        option_type=option_type,
    )["replication_error"].abs().max()

    rn_replication_gap = abs(
        price_by_risk_neutral_expectation(
            market=market,
            strike=strike,
            option_type=option_type,
        )
        - solve_replicating_portfolio(
            market=market,
            strike=strike,
            option_type=option_type,
        ).option_price
    )

    real_probability_matches = int(
        real_probability_audit["matches_no_arbitrage_price"].sum()
    )

    parity_errors = parity_audit.loc[
        parity_audit["check"].isin(
            [
                "difference_form_error",
                "portfolio_form_error",
            ]
        ),
        "value",
    ].abs()

    two_period_gap = abs(
        two_period_expectation_price
        - two_period_price.value_0
    )

    n_period_one_step_gap = abs(
        one_period_general.price
        - portfolio.option_price
    )

    n_period_two_step_gap = abs(
        two_period_general.price
        - two_period_price.value_0
    )

    crr_error_improved = (
        convergence_audit["abs_error_vs_reference"].iloc[-1]
        < convergence_audit["abs_error_vs_reference"].iloc[0]
    )

    rows = [
        {
            "check": "one_period_replication_exact",
            "value": one_period_replication_error,
            "pass_condition": f"value < {TOL}",
            "passed": one_period_replication_error < TOL,
        },
        {
            "check": "risk_neutral_price_equals_replication_price",
            "value": rn_replication_gap,
            "pass_condition": f"value < {TOL}",
            "passed": rn_replication_gap < TOL,
        },
        {
            "check": "real_world_probability_not_pricing_input",
            "value": real_probability_matches,
            "pass_condition": "only p = q matches no-arbitrage price",
            "passed": real_probability_matches == 1,
        },
        {
            "check": "put_call_parity",
            "value": parity_errors.max(),
            "pass_condition": f"value < {TOL}",
            "passed": parity_errors.max() < TOL,
        },
        {
            "check": "two_period_backward_induction_equals_terminal_expectation",
            "value": two_period_gap,
            "pass_condition": f"value < {TOL}",
            "passed": two_period_gap < TOL,
        },
        {
            "check": "n_period_model_matches_one_period_case",
            "value": n_period_one_step_gap,
            "pass_condition": f"value < {TOL}",
            "passed": n_period_one_step_gap < TOL,
        },
        {
            "check": "n_period_model_matches_two_period_case",
            "value": n_period_two_step_gap,
            "pass_condition": f"value < {TOL}",
            "passed": n_period_two_step_gap < TOL,
        },
        {
            "check": "time_scaled_crr_moves_toward_black_scholes_reference",
            "value": convergence_audit["abs_error_vs_reference"].iloc[-1],
            "pass_condition": "final error < initial error",
            "passed": crr_error_improved,
        },
    ]

    return pd.DataFrame(rows)


audit = final_notebook_audit()

display(audit)

notebook_passed = bool(audit["passed"].all())

assert notebook_passed

print("Final notebook audit passed.")
print("Core result: derivative price was derived from no-arbitrage replication, not real-world forecasting.")

,check,value,pass_condition,passed
0,one_period_replication_exact,0.000000,value < 1e-10,True
1,risk_neutral_price_equals_replication_price,0.000000,value < 1e-10,True
2,real_world_probability_not_pricing_input,1.000000,only p = q matches no-arbitrage price,True
3,put_call_parity,0.000000,value < 1e-10,True
4,two_period_backward_induction_equals_terminal_...,0.000000,value < 1e-10,True
5,n_period_model_matches_one_period_case,0.000000,value < 1e-10,True
6,n_period_model_matches_two_period_case,0.000000,value < 1e-10,True
7,time_scaled_crr_moves_toward_black_scholes_ref...,0.003905,final error < initial error,True


Final notebook audit passed.
Core result: derivative price was derived from no-arbitrage replication, not real-world forecasting.


## 11. Notebook conclusion

This notebook established the first principle of derivative pricing:

> A derivative price is determined by no-arbitrage replication, not by forecasting the real-world direction of the underlying asset.

The one-period binomial model showed that an option payoff can be replicated exactly using:

- stock holdings
- risk-free borrowing or lending

The cost of that replicating portfolio is the no-arbitrage option price.

The risk-neutral probability was then introduced as an equivalent way to express the same replication price. It was not used as a real-world forecast.

The notebook also verified:

- the no-arbitrage condition $d < R < u$
- exact one-period replication
- equality between replication pricing and risk-neutral pricing
- separation between real-world probability and pricing probability
- arbitrage when the market option price differs from the replication price
- put-call parity
- two-period backward induction
- general $N$-period backward induction
- time-scaled CRR convergence toward the Black-Scholes reference price

## What this notebook proves

This notebook proves, in a discrete binomial setting, that derivative pricing can be built from replication and no-arbitrage.

It also shows how repeated one-period pricing leads naturally to backward induction.

## What this notebook does not prove

This notebook does not derive Black-Scholes.

It does not prove Ito's lemma.

It does not model Brownian motion.

It does not calibrate to real option-chain data.

It does not justify volatility surfaces, Heston, Bates, or jump-diffusion models.

Those topics require additional theoretical layers.

## Handoff to the next notebook

The next notebook should move from discrete binomial price changes to continuous random price paths.

The next topic is:

> Brownian motion, geometric Brownian motion, and quadratic variation.

That is the bridge from binomial replication toward continuous-time Black-Scholes pricing.